#Create Date Dimention Table

In [0]:
query = """
            WITH bounds AS (
                SELECT 
                    MAKE_DATE(YEAR(MIN(order_date)), 1, 1) AS start_date,
                    MAKE_DATE(YEAR(MAX(order_date)), 12, 31) AS end_date
                FROM silver.transtractions
            )
            SELECT 
                ROW_NUMBER() OVER (ORDER BY d) AS date_key,
                d AS full_day,
                day(d) AS day,
                date_format(d, 'EEEE') AS day_name,
                weekofyear(d) AS week,
                month(d) AS month,
                date_format(d, 'MMMM') AS month_name,
                quarter(d) AS quarter,
                year(d) AS year,
                CASE WHEN dayofweek(d) IN (1,7) THEN 1 ELSE 0 END AS is_weekend
            FROM bounds
            LATERAL VIEW explode(sequence(start_date, end_date, INTERVAL 1 DAY)) t AS d
"""
df = spark.sql(query)


#Write into gold table

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_date")
)

In [0]:
%sql
SELECT * FROM gold.dim_date